<a href="https://colab.research.google.com/github/ransa17/DSA-Practice/blob/main/Hindi_OCR_Label_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hindi & Multi-Script Legal Document OCR — Label Prep Notebook
**Scope: Phase 1 / Day 2–3 — turn your (PDF + manual correction) pairs into OCR fine-tuning data.**
Runs after `Hindi_OCR_Data_Gathering.ipynb` — it assumes `PROJECT_ROOT` already exists on your Drive. This notebook does not train anything; it produces `training_data.jsonl`, ready for the Track A / Track B fine-tuning notebook.

## The convention — read this before writing any correction files
For every document, you provide **two files with the same base name**:
```
raw_docs/
  doc001.pdf     <- the scanned document
  doc001.md      <- your manual correction (or doc001.docx — Word works too)
```
The correction file is structured with one heading per region:
```markdown
# doc001

## text
<full body text, corrected, in reading order>

## signature
<transcribed signature text, or "[illegible]", or "[none]">

## stamp
<stamp text, or "[illegible]", or "[none]">

## date
<corrected date text>
```
- **Multiple dates / signatures / text blocks?** Just repeat the heading with a number: `## date_2`, `## signature_2`, `## text_2` — each becomes its own region, in order.
- **.docx version:** same idea, but use real Word heading styles (Heading 2) instead of `##` — everything else is identical.
- Recognized region classes: `text`, `signature`, `stamp`, `date` (a few synonyms like `body`/`seal`/`sign` are auto-mapped). Anything else is kept but flagged **unrecognized_class** in the report so you can catch a typo before it trains on the wrong label.
- **Nothing here fabricates a bounding box.** If a document matches one of the 20 templates (once its field map exists), you get the full region+box grounding format for free. Freeform notesheets train on the plain corrected-text target for now — real, usable, just without pixel coordinates until the template/labeling pipeline catches up.

## Cell 1 — Mount Drive + confirm the project structure exists

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/hindi_legal_ocr"
import os
assert os.path.isdir(PROJECT_ROOT), f"{PROJECT_ROOT} not found — run Hindi_OCR_Data_Gathering.ipynb first."

RAW_DOCS_DIR = f"{PROJECT_ROOT}/raw_docs"
OUT_DIR = f"{PROJECT_ROOT}/real_labeled/prepared"
os.makedirs(RAW_DOCS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("Drop your (doc.pdf + doc.md/.docx) pairs into:")
print(" ", RAW_DOCS_DIR)
print("(Easiest way: open that folder in Drive's web UI or the Drive desktop app, and drag files in.)")

Mounted at /content/drive
Drop your (doc.pdf + doc.md/.docx) pairs into:
  /content/drive/MyDrive/hindi_legal_ocr/raw_docs
(Easiest way: open that folder in Drive's web UI or the Drive desktop app, and drag files in.)


## Cell 2 — Install dependencies

In [2]:
%%capture
!apt-get -qq install -y poppler-utils
!pip install -q pdf2image python-docx pandas

## Cell 3 — The converter
Same script tested and delivered alongside this notebook (`training_pair_builder.py`). Writing it here via `%%writefile` keeps this notebook fully self-contained — no separate upload needed.

In [3]:
%%writefile training_pair_builder.py
"""
Converts (document.pdf + document.md/.docx) pairs into OCR fine-tuning training data.

Convention (matches what was agreed with the user):
  raw_docs/
    doc001.pdf     <- the scanned document
    doc001.md      <- your manual correction, SAME base filename, using this structure:

      # doc001

      ## text
      <full body text, corrected, in reading order>

      ## signature
      <transcribed signature text, or "[illegible]", or "[none]" if there isn't one>

      ## stamp
      <stamp text, or "[illegible]", or "[none]">

      ## date
      <corrected date text>

  Multiple dates / signatures / text blocks: just repeat the heading with a number,
  e.g. "## date_2", "## signature_2", "## text_2" — each becomes its own region, in order.
  Recognized classes (case-insensitive, numeric suffix stripped): text, signature, stamp, date.
  Anything else is kept but flagged "unrecognized_class" in the report so you can fix a typo.

  .docx correction files use the same convention, except headings are real Word heading
  styles ("Heading 1"/"Heading 2"/etc.) instead of "##" markdown.

What this produces, per document:
  images/<doc_id>.png              - the page rendered as an image (training needs images, not PDFs)
  parsed/<doc_id>.json             - structured {"regions": [{"class", "seq", "text"}, ...]}
  training_plain/<doc_id>.txt      - a single corrected-text training target (works today, no
                                      bounding boxes needed) — every region's text, with
                                      signature/stamp/date inline-labelled, in document order
  training_data.jsonl              - the full manifest, one line per document, ready for SFT:
                                      {"image": ..., "prompt": "Free OCR.", "target": ...}
  report.csv                       - per-document status: region counts, any unrecognized
                                      class names, whether a template field map (bounding boxes)
                                      was found for it

Bounding-box / grounding-format output (<|ref|>class<|/ref|><|det|>[[x1,y1,x2,y2]]<|/det|>) is
added ONLY for documents matched to a template field map with real coordinates — see
`try_template_boxes()`. Freeform notesheets have no such map yet (Day 2-3 work), so they train
via the plain-text target for now rather than fabricated boxes. Nothing here invents a
coordinate — that would poison the training set silently, which is exactly what the plan's
"no silent caps / never fabricate accuracy" rule is against.
"""

import os
import re
import json
import csv
import sys
from pathlib import Path

try:
    from pdf2image import convert_from_path
except ImportError:
    convert_from_path = None

try:
    import docx as python_docx
except ImportError:
    python_docx = None


RECOGNIZED_CLASSES = {"text", "signature", "stamp", "date"}
CLASS_SYNONYMS = {
    "body": "text", "paragraph": "text", "para": "text", "content": "text",
    "sign": "signature", "signatures": "signature",
    "seal": "stamp", "stamps": "stamp",
    "dates": "date",
}


def normalize_class(raw_heading):
    """'## date_2' -> ('date', 2); '## Signature' -> ('signature', 1); '## Notes' -> ('notes', 1, unrecognized)"""
    h = raw_heading.strip().lower()
    m = re.match(r"^(.*?)[\s_]*(\d+)?$", h)
    base = m.group(1).strip() if m else h
    seq = int(m.group(2)) if m and m.group(2) else 1
    base = CLASS_SYNONYMS.get(base, base)
    recognized = base in RECOGNIZED_CLASSES
    return base, seq, recognized


def parse_markdown_correction(path):
    text = Path(path).read_text(encoding="utf-8")
    lines = text.splitlines()
    regions = []
    current = None
    for line in lines:
        m = re.match(r"^##\s+(.+?)\s*$", line)
        if m:
            if current is not None:
                regions.append(current)
            base, seq, recognized = normalize_class(m.group(1))
            current = {"class": base, "seq": seq, "raw_heading": m.group(1).strip(),
                       "recognized": recognized, "text_lines": []}
        elif re.match(r"^#\s+", line):
            continue  # top-level "# doc_id" title line — not a region
        else:
            if current is not None:
                current["text_lines"].append(line)
    if current is not None:
        regions.append(current)
    for r in regions:
        r["text"] = "\n".join(r.pop("text_lines")).strip()
    return regions


def parse_docx_correction(path):
    if python_docx is None:
        raise RuntimeError("python-docx not installed — cannot parse .docx correction files")
    doc = python_docx.Document(str(path))
    regions = []
    current = None
    for para in doc.paragraphs:
        style = (para.style.name or "").lower()
        heading_level_match = re.match(r"^heading (\d+)$", style)
        is_title = style.startswith("title")
        is_region_heading = heading_level_match and int(heading_level_match.group(1)) >= 2
        if is_title or (heading_level_match and not is_region_heading):
            continue  # top-level title / "Heading 1" doc-id line — not a region, same as a single "#" in markdown
        if is_region_heading:
            if current is not None:
                regions.append(current)
            base, seq, recognized = normalize_class(para.text)
            current = {"class": base, "seq": seq, "raw_heading": para.text.strip(),
                       "recognized": recognized, "text_lines": []}
        else:
            if current is not None:
                current["text_lines"].append(para.text)
    if current is not None:
        regions.append(current)
    for r in regions:
        r["text"] = "\n".join(r.pop("text_lines")).strip()
    return regions


def parse_correction_file(path):
    path = Path(path)
    if path.suffix.lower() == ".md":
        return parse_markdown_correction(path)
    elif path.suffix.lower() == ".docx":
        return parse_docx_correction(path)
    else:
        raise ValueError(f"Unsupported correction file type: {path.suffix} ({path})")


def render_pdf_to_images(pdf_path, out_dir, doc_id, dpi=200):
    """Renders each page. Returns list of (page_num, image_path)."""
    if convert_from_path is None:
        raise RuntimeError("pdf2image not installed — cannot render PDFs to images")
    pages = convert_from_path(str(pdf_path), dpi=dpi)
    out = []
    for i, page in enumerate(pages, start=1):
        suffix = "" if len(pages) == 1 else f"_p{i}"
        img_path = Path(out_dir) / f"{doc_id}{suffix}.png"
        page.save(img_path, "PNG")
        out.append((i, str(img_path)))
    return out


def try_template_boxes(doc_id, regions, template_field_maps_dir, template_assignment):
    """
    If a template field map exists for this document (template_assignment maps doc_id ->
    template_name, built by the Day 2-3 template classifier), look up each region's
    bounding box by matching region class/heading to a field-map key. Returns
    (regions_with_boxes, matched: bool) — matched=False leaves every box as None,
    never a guessed one.
    """
    if template_field_maps_dir is None or template_assignment is None:
        return regions, False
    template_name = template_assignment.get(doc_id)
    if not template_name:
        return regions, False
    map_path = Path(template_field_maps_dir) / f"{template_name}.json"
    if not map_path.exists():
        return regions, False
    field_map = json.loads(map_path.read_text(encoding="utf-8"))
    matched_any = False
    for r in regions:
        key_candidates = [r["raw_heading"], r["class"], f"{r['class']}_{r['seq']}"]
        box = None
        for k in key_candidates:
            if k in field_map:
                box = field_map[k]
                break
        r["box"] = box
        if box:
            matched_any = True
    return regions, matched_any


def assemble_plain_text(regions):
    """Full corrected-text training target — no coordinates needed, usable today."""
    parts = []
    for r in regions:
        text = r["text"]
        if not text:
            continue
        if r["class"] == "text":
            parts.append(text)
        elif r["class"] in ("signature", "stamp", "date"):
            label = r["class"].capitalize()
            suffix = f" {r['seq']}" if r["seq"] > 1 else ""
            parts.append(f"[{label}{suffix}: {text}]")
        else:
            parts.append(f"[{r['raw_heading']}: {text}]")
    return "\n".join(parts)


def assemble_grounding_text(regions):
    """DeepSeek-OCR grounding format — only meaningful for regions that have a real box."""
    lines = []
    for r in regions:
        box = r.get("box")
        if not box:
            continue
        lines.append(f"<|ref|>{r['class']}<|/ref|><|det|>[{box}]<|/det|>{r['text']}")
    return "\n".join(lines)


def build_dataset(raw_docs_dir, out_root, template_field_maps_dir=None, template_assignment=None, dpi=200):
    raw_docs_dir = Path(raw_docs_dir)
    out_root = Path(out_root)
    images_dir = out_root / "images"
    parsed_dir = out_root / "parsed"
    plain_dir = out_root / "training_plain"
    for d in (images_dir, parsed_dir, plain_dir):
        d.mkdir(parents=True, exist_ok=True)

    pdfs = {p.stem: p for p in raw_docs_dir.glob("*.pdf")}
    corrections = {}
    for ext in ("*.md", "*.docx"):
        for p in raw_docs_dir.glob(ext):
            corrections.setdefault(p.stem, p)

    report_rows = []
    manifest_lines = []

    all_ids = sorted(set(pdfs) | set(corrections))
    for doc_id in all_ids:
        row = {"doc_id": doc_id, "status": "", "n_regions": 0, "unrecognized_classes": "",
               "has_boxes": False, "pages": 0}
        pdf_path = pdfs.get(doc_id)
        corr_path = corrections.get(doc_id)

        if pdf_path is None:
            row["status"] = "MISSING_PDF"
            report_rows.append(row)
            continue
        if corr_path is None:
            row["status"] = "MISSING_CORRECTION_FILE"
            report_rows.append(row)
            continue

        try:
            pages = render_pdf_to_images(pdf_path, images_dir, doc_id, dpi=dpi)
        except Exception as e:
            row["status"] = f"PDF_RENDER_ERROR: {e}"
            report_rows.append(row)
            continue
        row["pages"] = len(pages)

        try:
            regions = parse_correction_file(corr_path)
        except Exception as e:
            row["status"] = f"CORRECTION_PARSE_ERROR: {e}"
            report_rows.append(row)
            continue

        if not regions:
            row["status"] = "NO_REGIONS_FOUND — check the file uses '## heading' (md) or Word Heading styles (docx)"
            report_rows.append(row)
            continue

        unrecognized = [r["raw_heading"] for r in regions if not r["recognized"]]
        regions, has_boxes = try_template_boxes(doc_id, regions, template_field_maps_dir, template_assignment)

        parsed_out = {"doc_id": doc_id, "pages": len(pages), "regions": regions}
        (parsed_dir / f"{doc_id}.json").write_text(
            json.dumps(parsed_out, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        plain_text = assemble_plain_text(regions)
        (plain_dir / f"{doc_id}.txt").write_text(plain_text, encoding="utf-8")

        image_path = pages[0][1]  # single-page assumption for the manifest; see docstring
        manifest_lines.append(json.dumps({
            "doc_id": doc_id,
            "image": str(Path(image_path).relative_to(out_root)),
            "prompt": "Free OCR.",
            "target": plain_text,
            "grounding_target": assemble_grounding_text(regions) if has_boxes else None,
        }, ensure_ascii=False))

        row["status"] = "OK"
        row["n_regions"] = len(regions)
        row["unrecognized_classes"] = "; ".join(unrecognized)
        row["has_boxes"] = has_boxes
        report_rows.append(row)

    (out_root / "training_data.jsonl").write_text("\n".join(manifest_lines) + ("\n" if manifest_lines else ""), encoding="utf-8")

    with open(out_root / "report.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["doc_id", "status", "n_regions", "unrecognized_classes", "has_boxes", "pages"])
        writer.writeheader()
        writer.writerows(report_rows)

    ok = sum(1 for r in report_rows if r["status"] == "OK")
    print(f"Processed {len(report_rows)} document(s): {ok} OK, {len(report_rows) - ok} need attention.")
    print(f"Manifest: {out_root / 'training_data.jsonl'}  ({len(manifest_lines)} training pairs)")
    print(f"Report:   {out_root / 'report.csv'}")
    return report_rows


if __name__ == "__main__":
    raw_dir = sys.argv[1] if len(sys.argv) > 1 else "raw_docs"
    out_dir = sys.argv[2] if len(sys.argv) > 2 else "."
    build_dataset(raw_dir, out_dir)

Writing training_pair_builder.py


## Cell 4 — Run the converter
Processes every (pdf + md/docx) pair found in `raw_docs/`. Safe to re-run — it just reprocesses whatever's there; nothing upstream gets deleted.

In [4]:
import importlib, sys
sys.path.insert(0, '.')
import training_pair_builder
importlib.reload(training_pair_builder)

# --- Optional: if you've already built template field maps (Day 2 template classifier work),
#     point these at them and provide a {doc_id: template_name} assignment to unlock the
#     grounding/bounding-box format for template-matched documents. Leave as None for now.
TEMPLATE_FIELD_MAPS_DIR = None   # e.g. f"{PROJECT_ROOT}/template_field_maps"
TEMPLATE_ASSIGNMENT = None       # e.g. {"doc007": "Template_03"}

report_rows = training_pair_builder.build_dataset(
    raw_docs_dir=RAW_DOCS_DIR,
    out_root=OUT_DIR,
    template_field_maps_dir=TEMPLATE_FIELD_MAPS_DIR,
    template_assignment=TEMPLATE_ASSIGNMENT,
)

Processed 0 document(s): 0 OK, 0 need attention.
Manifest: /content/drive/MyDrive/hindi_legal_ocr/real_labeled/prepared/training_data.jsonl  (0 training pairs)
Report:   /content/drive/MyDrive/hindi_legal_ocr/real_labeled/prepared/report.csv


## Cell 5 — Review the report
Anything that isn't `OK` needs your attention before it's used for training — a missing pairing file, a parse error, or an unrecognized heading (likely a typo).

In [5]:
import pandas as pd
report = pd.read_csv(f"{OUT_DIR}/report.csv")
display(report)

not_ok = report[report["status"] != "OK"]
if len(not_ok):
    print(f"\n{len(not_ok)} document(s) need attention:")
    display(not_ok)
else:
    print("\nAll documents processed cleanly.")

flagged = report[report["unrecognized_classes"].fillna("") != ""]
if len(flagged):
    print(f"\n{len(flagged)} document(s) have an unrecognized heading — check for typos:")
    display(flagged[["doc_id", "unrecognized_classes"]])

n_boxed = report["has_boxes"].sum()
print(f"\n{n_boxed} / {len(report)} documents have real bounding boxes (template-matched).")
print(f"The rest train on the plain-text target — still valid, just not region-grounded yet.")

,doc_id,status,n_regions,unrecognized_classes,has_boxes,pages



All documents processed cleanly.

0 / 0 documents have real bounding boxes (template-matched).
The rest train on the plain-text target — still valid, just not region-grounded yet.


## Cell 6 — Peek at one training pair
Sanity check before this feeds fine-tuning — read one manifest line and its target text side by side with the source image.

In [6]:
import json
from PIL import Image

lines = open(f"{OUT_DIR}/training_data.jsonl", encoding="utf-8").read().splitlines()
if lines:
    example = json.loads(lines[0])
    print("doc_id:", example["doc_id"])
    print("target text:\n", example["target"])
    print("\ngrounding target:\n", example["grounding_target"] or "(none — no template box match)")
    display(Image.open(f"{OUT_DIR}/{example['image']}"))
else:
    print("No training pairs yet — add files to raw_docs/ and rerun Cell 4.")

No training pairs yet — add files to raw_docs/ and rerun Cell 4.


## Summary
- `training_data.jsonl` in `real_labeled/prepared/` is the manifest the fine-tuning notebook reads — one line per document, image path + corrected target text (+ grounding format where boxes exist).
- Rerunning this notebook after adding more (pdf + correction) pairs to `raw_docs/` picks up everything, old and new — nothing needs to be redone.
- Next: once the Day 2 template classifier + field maps exist, point `TEMPLATE_FIELD_MAPS_DIR`/`TEMPLATE_ASSIGNMENT` at them in Cell 4 and rerun — template-matched documents automatically upgrade from plain-text targets to full region+box grounding targets, no correction files need to be rewritten.